In [ ]:
import json
import redis
from sentence_transformers import SentenceTransformer, util
from fuzzywuzzy import fuzz
import re
import string
import numpy as np
import time

# ---------------------- Text Normalization & Cleaning ---------------------- #
def normalize_text(text):
    """Lowercase and remove punctuation and extra spaces."""
    translator = str.maketrans('', '', string.punctuation)
    return text.lower().translate(translator).strip()

def remove_quantity_info(text):
    """Remove quantity/unit information like '1 ltr', '500 ml', 'pack of 3', etc."""
    text = re.sub(r'\d+\s*(ltr|ml|g|kg|litre|milliliter|pieces|piece|x\s*\d+)', '', text)
    text = re.sub(r'\b\d+\b', '', text)
    return text

def synonym_replace(text):
    """Replace or remove domain-specific synonyms and filler words."""
    remove_terms = [
        "homogenised", "homogenized", "pasteurised", "pasteurized",
        "uht", "tetra pack", "tetra", "carton", "value pack", "fresh",
        "standardised", "standardized"
    ]
    for term in remove_terms:
        text = re.sub(r'\b' + re.escape(term) + r'\b', '', text)
    return text

def clean_product_name(name):
    """Cleans a product name by normalizing text, removing quantity/unit info, and applying synonym replacement."""
    name = normalize_text(name)
    name = remove_quantity_info(name)
    name = synonym_replace(name)
    return re.sub(r'\s+', ' ', name).strip()

def composite_text(product):
    """Build composite text for matching by combining the cleaned product name and its MRP."""
    cleaned_name = clean_product_name(product['name'])
    return f"{cleaned_name} {product['mrp']}"

# ------------------------- Matching Helpers ------------------------- #
def mrp_match(a, b, error=4):
    """Check if two MRPs are within a small tolerance or an integer multiple."""
    if abs(a - b) <= error:
        return True
    smaller = min(a, b)
    larger = max(a, b)
    if smaller == 0:
        return False
    candidate_multiple = round(larger / smaller)
    if abs(candidate_multiple * smaller - larger) <= error:
        return True
    return False

def fuzzy_brand_match(brand1, brand2, threshold=70):
    """Returns True if fuzzy matching score between two brand names is at least 'threshold'."""
    score = fuzz.token_set_ratio(brand1, brand2)
    return score >= threshold

# ------------------------- Semantic Matching ------------------------- #
def map_unmapped_products(unmapped_products, similarity_threshold=0.75):
    """Match unmapped products against each other using semantic similarity."""
    if len(unmapped_products) < 2:
        print("Not enough unmapped products to match (need at least 2).")
        return []

    model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
    unmapped_texts = [composite_text(prod) for prod in unmapped_products]
    embeddings = model.encode(unmapped_texts, convert_to_tensor=True)

    mappings = []
    for i, prod1 in enumerate(unmapped_products):
        for j, prod2 in enumerate(unmapped_products[i+1:], start=i+1):
            if prod1['prices'][0]['source'] == prod2['prices'][0]['source']:
                continue

            if not fuzzy_brand_match(prod1['brand'], prod2['brand']) or not mrp_match(prod1['mrp'], prod2['mrp']):
                continue

            embedding1 = embeddings[i:i+1]
            embedding2 = embeddings[j:j+1]
            cosine_score = util.cos_sim(embedding1, embedding2)[0][0].item()

            if cosine_score >= similarity_threshold:
                # Extract variant details (assuming first variant is representative)
                prod1_variant = prod1['prices'][0]['variants'][0]
                prod2_variant = prod2['prices'][0]['variants'][0]
                
                mappings.append({
                    "source": {
                        "id": prod1['id'],
                        "name": prod1['name'],
                        "brand": prod1['brand'],
                        "mrp": prod1['mrp'],
                        "source": prod1['prices'][0]['source'],
                        "weight": prod1_variant['weight'],
                        "deeplink": prod1_variant['deeplink'],
                        "image_url": prod1_variant.get('image_url', '')  # Add image_url, default to empty string if missing
                    },
                    "match": {
                        "id": prod2['id'],
                        "name": prod2['name'],
                        "brand": prod2['brand'],
                        "mrp": prod2['mrp'],
                        "source": prod2['prices'][0]['source'],
                        "weight": prod2_variant['weight'],
                        "deeplink": prod2_variant['deeplink'],
                        "image_url": prod2_variant.get('image_url', '')  # Add image_url, default to empty string if missing
                    },
                    "similarity": round(cosine_score, 4)
                })

    return mappings

# ------------------------- Worker Logic ------------------------- #
def process_unmapped_from_redis():
    # Connect to Redis
    redis_client = redis.Redis(host='localhost', port=6379, db=0, decode_responses=True)

    # Process each Redis key matching 'unmapped:*'
    for key in redis_client.keys("unmapped:*"):
        query = key.split(":", 1)[1]  # Extract query from key (e.g., "kurkure")
        unmapped_json = redis_client.get(key)
        if not unmapped_json:
            continue

        unmapped_products = json.loads(unmapped_json)
        print(f"Processing {len(unmapped_products)} unmapped products for query '{query}'")

        # Run semantic matching on unmapped products
        mappings = map_unmapped_products(unmapped_products)

        if mappings:
            # Wrap mappings in a top-level structure with query
            result = {
                "query": query,
                "mappings": mappings
            }
            mappings_json = json.dumps(result)
            redis_key = f"pending_mappings:{query}"
            redis_client.set(redis_key, mappings_json, ex=24*3600)  # 24-hour expiration
            print(f"Stored {len(mappings)} mappings for query '{query}' in Redis under '{redis_key}'")
            # Do NOT delete the unmapped key to preserve it
        else:
            print(f"No mappings found for query '{query}'")

if __name__ == "__main__":
    while True:
        process_unmapped_from_redis()
        time.sleep(60)  # Run every 60 seconds

Processing 50 unmapped products for query 'ku'
No mappings found for query 'ku'
Processing 83 unmapped products for query 'kurkure'
Stored 16 mappings for query 'kurkure' in Redis under 'pending_mappings:kurkure'
Processing 101 unmapped products for query 'condom'
Stored 36 mappings for query 'condom' in Redis under 'pending_mappings:condom'
Processing 82 unmapped products for query 'con'
No mappings found for query 'con'


KeyboardInterrupt: 

In [1]:
pip install sentence-transformers fuzzywuzzy redis

Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 340 kB 9.4 MB/s eta 0:00:01
     |████████████████████████████████| 261 kB 113.4 MB/s eta 0:00:01
     |████████████████████████████████| 10.4 MB 117.1 MB/s eta 0:00:01
     |████████████████████████████████| 66.5 MB 20.7 MB/s eta 0:00:01
     |████████████████████████████████| 481 kB 31.4 MB/s eta 0:00:01
     |████████████████████████████████| 11.1 MB 20.3 MB/s eta 0:00:01
     |████████████████████████████████| 30.3 MB 21.9 MB/s eta 0:00:01
     |████████████████████████████████| 78 kB 30.9 MB/s eta 0:00:01
     |████████████████████████████████| 172 kB 39.0 MB/s eta 0:00:01
     |████████████████████████████████| 194 kB 32.0 MB/s eta 0:00:01
     |████████████████████████████████| 1.6 MB 45.9 MB/s eta 0:00:01
     |████████████████████████████████| 134 kB 39.5 MB/s eta 0:00:01
     |████████████████████████████████| 6.2 MB 55.9 MB/s eta 0:00:01
     |███████████████